# URECA: NSH / DEUDF / MPF reconstruction

**Runtime -> Change runtime type -> T4 GPU** before you do anything else.

Order of work:
1. Upload `ureca3d.zip` (cell 1)
2. Run the unit tests (cell 3) until they pass -- do this *before* training
3. Train (cells 4-6)
4. Build the comparison table and figures (cells 7-8)


In [ ]:
# 1. upload the zip, unpack, install
from google.colab import files
up = files.upload()          # choose ureca3d.zip
!unzip -qo ureca3d.zip -d /content && cd /content/ureca3d && ls
!pip -q install trimesh scikit-image pytest

In [ ]:
# 2. check the GPU is really there
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
%cd /content/ureca3d

In [ ]:
# 3. property tests -- run these before any long training job
!python -m pytest tests -q

## Training
On a T4, 10k iterations at batch 15000 takes roughly 15-25 min for NSH/DEUDF and
a bit longer for MPF (two heads, more loss terms). Start with `--iters 3000` to
confirm the shape looks right, *then* run the full thing.

Add `--no_sphere_init` to the MPF commands to reproduce the empty-mesh failure
described in the README.

In [ ]:
# 4. NSH -- watertight shapes are its home turf
!python train.py --method nsh --shape torus      --points 20000 --iters 10000 --res 192
!python train.py --method nsh --shape cube_sheet --points 20000 --iters 10000 --res 192   # should struggle on the sheet
!python train.py --method nsh --shape torus      --points 1000  --iters 10000 --res 192   # sparse-input claim

In [ ]:
# 5. DEUDF -- open surfaces
!python train.py --method deudf --shape thin_plate --points 20000 --iters 10000 --res 192
!python train.py --method deudf --shape cube_sheet --points 20000 --iters 10000 --res 192
!python train.py --method deudf --shape thin_plate --points 20000 --iters 10000 --noise 0.0025 --res 192

In [ ]:
# 6. MPF -- the money shot is cube_sheet: watertight cube + non-manifold thin sheet
!python train.py --method mpf --shape cube_sheet --points 20000 --iters 10000 --res 192
!python train.py --method mpf --shape thin_plate --points 20000 --iters 10000 --res 192
!python train.py --method mpf --shape torus      --points 20000 --iters 10000 --res 192

In [ ]:
# 7. collect every metrics.json into one markdown table
import glob, json
from common.metrics import format_table
rows = [json.load(open(p)) for p in sorted(glob.glob('out/*/metrics.json'))]
tbl = format_table(rows)
print(tbl)
open('out/results.md','w').write(tbl)

In [ ]:
# 8. render every reconstruction to a PNG grid -- this is what you send Prof. He
import glob, os, numpy as np, trimesh, matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def show(ax, path, title):
    ax.set_title(title, fontsize=8); ax.set_axis_off()
    if not os.path.exists(path): ax.text(0,0,0,'missing'); return
    m = trimesh.load(path, process=False)
    if len(m.faces) > 60000:
        m = m.simplify_quadric_decimation(60000) if hasattr(m,'simplify_quadric_decimation') else m
    tri = m.vertices[m.faces]
    pc = Poly3DCollection(tri, facecolor='#c9ced6', edgecolor='none', alpha=1.0)
    ax.add_collection3d(pc)
    ax.set_xlim(-1,1); ax.set_ylim(-1,1); ax.set_zlim(-1,1)
    ax.view_init(elev=22, azim=38)

dirs = sorted(glob.glob('out/*'))
n = len(dirs); cols = 4; rows_ = (n+cols-1)//cols
fig = plt.figure(figsize=(3.1*cols, 3.1*rows_))
for i,d in enumerate(dirs):
    ax = fig.add_subplot(rows_, cols, i+1, projection='3d')
    show(ax, os.path.join(d,'recon.ply'), os.path.basename(d))
fig.tight_layout(); fig.savefig('out/gallery.png', dpi=150)
print('written out/gallery.png')

In [ ]:
# 9. download everything
!zip -qr results.zip out && echo ok
from google.colab import files; files.download('results.zip')